In [1]:
!git clone https://github.com/Treften/ad-icrl-main.git

Cloning into 'ad-icrl-main'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 277 (delta 170), reused 198 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 22.98 MiB | 11.60 MiB/s, done.
Resolving deltas: 100% (170/170), done.


In [2]:
!nvidia-smi

Sun Oct 11 16:30:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%cd /kaggle/working/ad-icrl-main

/kaggle/working/ad-icrl-main


In [4]:
!git pull

Already up to date.


In [5]:
!ls

notebooks  README.md  requirements  saved_data	scripts  src  static


In [6]:
!ls /kaggle/working

ad-icrl-main


In [7]:
!pip install -r requirements/requirements_colab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 11.6 MB/s eta 0:00:00


In [8]:
import sys
sys.path.append("/kaggle/working/ad-icrl-main")

In [9]:
!mkdir -p /kaggle/working/data

!tar -xvzf ../../input/Dark-Room_PPO/learning-history-ppo-01.tar.gz -C /kaggle/working/data
!tar -xvzf /kaggle/input/Dark-Room_PPO/learning-history-ppo-02.tar.gz -C /kaggle/working/data
!tar -xvzf /kaggle/input/Dark-Room_PPO/learning-history-ppo-03.tar.gz -C /kaggle/working/data

tar (child): ../../input/Dark-Room_PPO/learning-history-ppo-01.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now
tar (child): /kaggle/input/Dark-Room_PPO/learning-history-ppo-02.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now
tar (child): /kaggle/input/Dark-Room_PPO/learning-history-ppo-03.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now


In [10]:
%env CLEARML_WEB_HOST=https://app.clear.ml/
%env CLEARML_API_HOST=https://api.clear.ml
%env CLEARML_FILES_HOST=https://files.clear.ml
%env CLEARML_API_ACCESS_KEY=O5RLGEGONJ5LBI8PQIKSYMEROGNDMV
%env CLEARML_API_SECRET_KEY=hP6zY1rKMmEz25gYxDZZ_X767bco1vEedAVo2K5FLiKif6LKXjw3NJTCYMgOF0kNJuI

env: CLEARML_WEB_HOST=https://app.clear.ml/
env: CLEARML_API_HOST=https://api.clear.ml
env: CLEARML_FILES_HOST=https://files.clear.ml
env: CLEARML_API_ACCESS_KEY=O5RLGEGONJ5LBI8PQIKSYMEROGNDMV
env: CLEARML_API_SECRET_KEY=hP6zY1rKMmEz25gYxDZZ_X767bco1vEedAVo2K5FLiKif6LKXjw3NJTCYMgOF0kNJuI


In [11]:
!pip install clearml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.0 MB/s eta 0:00:0000:0100:01


In [12]:
%cd /kaggle/working/ad-icrl-main

/kaggle/working/ad-icrl-main


In [13]:
from clearml import Task, Logger
import os
import shutil

In [14]:
task = Task.get_task(task_id="6e49dff4026a4c74989bee249e5b1772")

artifact = task.artifacts["model_299999"]

path = artifact.get_local_copy()
dst_dir = "/kaggle/working/ad-icrl-main/saved_data/saved_models/m80_300000"
os.makedirs(dst_dir, exist_ok=True)

dst_path = os.path.join(dst_dir, os.path.basename(path))

shutil.copytree(path, dst_dir, dirs_exist_ok=True)
print(path)

████████████████████████████▋  99% | 530.00/534.83 MB [00:16<00:00, 33.97MB/s]: /usr/local/lib/python3.12/dist-packages/tqdm/std.py:636: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
████████████████████████████ 100% | 534.83/534.83 MB [00:16<-00:00, 33.03MB/s]: 


/root/.clearml/cache/storage_manager/global/78a0c63376166bdf8573cbc59be86565.darkroom_07-Oct-10-51-14_artifacts_archive_model_299999


In [ ]:
!DEVICE=cuda \
PYTHONPATH=/kaggle/working/ad-icrl-main \
python3 /kaggle/working/ad-icrl-main/src/dt/train.py \
--config.seq-len 60 \
--config.learning-rate 0.0003 \
--config.weight-decay 0.001 \
--config.embedding-dim 64 \
--config.hidden-dim 256 \
--config.warmup-steps 10000 \
--config.batch-size 128 \
--config.num-workers 4 \
--config.eval-freq 10000 \
--config.eval-episodes 30\
--config.num_updates 300000

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
cuda
4
300000
saved_data/saved_models/darkroom_11-Oct-16-31-56
ClearML Task: created new task id=dc4841befe3249949bdb6a0372140c9c
ClearML results page: https://app.clear.ml/projects/007c59b9dffd4616a5203a16ecd9d855/tasks/dc4841befe3249949bdb6a0372140c9c/output/log
['/kaggle/input/datasets/desfr1t/dark-room-ppo/saved_data/learning_history/ppo-01', '/kaggle/input/datasets/desfr1t/dark-room-ppo/saved_data/learning_history/ppo-02', '/kaggle/input/datasets/desfr1t/dark-room-ppo/saved_data/learning_hi

In [ ]:
!find /kaggle/input /kaggle/working -type d -name "ppo-01" 2>/dev/null

In [ ]:
root = "/kaggle/working/ad-icrl-main"

for current_dir, dirs, files in os.walk(root):
    if any("model" in f.lower() or "config" in f.lower() for f in files):
        print(f"\nDIR: {current_dir}")
        print([f for f in files if "model" in f.lower() or "config" in f.lower()])

In [ ]:
!PYTHONPATH=/kaggle/working/ad-icrl-main \
python3 /kaggle/working/ad-icrl-main/src/check_in_context/eval_one_model.py \
--model-dir  /kaggle/working/ad-icrl-main/saved_data/saved_models/m80_300000 \
--n-repeats 2 \
--eval-episodes 100 

In [ ]:
!ls /kaggle/working/ad-icrl-main/saved_data/saved_models/m300000

In [ ]:
from pathlib import Path

logs_dir = Path("/kaggle/working/ad-icrl-main/saved_data/saved_models/darkroom_14-Aug-15-40-38")

for file in logs_dir.iterdir():
    if file.is_file():
        print(file.name)